<a href="https://colab.research.google.com/github/paulovasantos-cmyk/seguranca-ia-bigdata/blob/main/isolation_forest_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Detecção de Anomalias em Logs de Autenticação
============================================
Modelo: Isolation Forest (Aprendizado Não Supervisionado)
Objetivo: Identificar comportamentos anômalos em tentativas de login (ex.: fora do horário,
          múltiplas falhas, IPs externos suspeitos).
Autores: Paulo Victor, Renato Lacerda e Rodrigo Pithon
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# ====================== CONFIGURAÇÕES ======================
np.random.seed(42)
plt.style.use('seaborn-v0_8')

# ====================== GERAÇÃO DE DADOS SINTÉTICOS ======================
n_samples = 5000

data = pd.DataFrame({
    'hour': np.random.randint(0, 24, n_samples),
    'failed_attempts': np.random.poisson(0.8, n_samples).astype(int),
    'is_weekend': np.random.choice([0, 1], n_samples),
    'external_ip': np.random.choice([0, 1], n_samples),
})

# Feature Engineering - Hora Cíclica (melhor representação)
data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)

# ====================== INJEÇÃO DE ANOMALIAS ======================
# Cria anomalias realistas (acessos noturnos com muitas falhas)
anomaly_indices = np.random.choice(range(4800, n_samples), size=80, replace=False)

data.loc[anomaly_indices, 'failed_attempts'] = np.random.randint(8, 15, size=len(anomaly_indices))
data.loc[anomaly_indices, 'hour'] = np.random.choice([2, 3, 4, 5], size=len(anomaly_indices))
data.loc[anomaly_indices, 'external_ip'] = 1

print(f"Total de amostras: {len(data)}")
print(f"Anomalias injetadas: {len(anomaly_indices)}")

# ====================== SELEÇÃO DE FEATURES ======================
features = ['hour_sin', 'hour_cos', 'failed_attempts', 'is_weekend', 'external_ip']

X = data[features].copy()

# Padronização (boa prática para Isolation Forest)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ====================== TREINAMENTO DO MODELO ======================
model = IsolationForest(
    n_estimators=200,
    contamination=0.02,      # Espera-se cerca de 2% de anomalias
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

# Treinamento
model.fit(X_scaled)

# Predição (-1 = anomalia, 1 = normal)
data['anomaly_score'] = model.decision_function(X_scaled)  # Quanto menor, mais anômalo
data['anomaly'] = model.predict(X_scaled)

# ====================== RESULTADOS ======================
anomalies = data[data['anomaly'] == -1]
print(f"\nAnomalias detectadas: {len(anomalies)}")
print("\nExemplos de anomalias detectadas:")
print(anomalies[['hour', 'failed_attempts', 'external_ip', 'anomaly_score']].head(10))

# ====================== VISUALIZAÇÃO ======================
plt.figure(figsize=(12, 6))

# Scatter Plot - Hora vs Tentativas Falhas
plt.subplot(1, 2, 1)
colors = ['blue' if x == 1 else 'red' for x in data['anomaly']]
plt.scatter(data['hour'], data['failed_attempts'], c=colors, alpha=0.7, s=15)
plt.title('Detecção de Anomalias - Isolation Forest')
plt.xlabel('Hora do Dia')
plt.ylabel('Tentativas Falhas')
plt.grid(True)

# Distribuição dos Scores
plt.subplot(1, 2, 2)
sns.histplot(data['anomaly_score'], bins=50, kde=True)
plt.axvline(0, color='red', linestyle='--', label='Limiar de Anomalia')
plt.title('Distribuição dos Scores de Anomalia')
plt.xlabel('Anomaly Score')
plt.legend()
plt.tight_layout()
plt.show()

# ====================== SALVAR RESULTADOS ======================
data.to_csv('resultados_anomalias_logins.csv', index=False)
print("\nResultados salvos em 'resultados_anomalias_logins.csv'")